# 00 — Smoke test (QUICK)

**Question.** Does every method train, evaluate, and produce latents end-to-end, and how long would the full run take?

Runs all methods (Tier A + Tier B) for seed 0 with 20 epochs into a separate cache
(`outputs/meeting_2026-10-14_quick`). Numbers here are **not results** — 20 epochs is far from converged.

In [1]:
RUN_TRAINING = True   # True: train runs that are not cached yet (cached runs are skipped)
QUICK = True          # True: 1 seed, 20 epochs, separate cache dir (smoke test)

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from get_shredded import meeting_plots as mp
from get_shredded.meeting_runner import (
    LATENT_METHODS, load_arrays, load_config, load_results, load_seed_info, prepare_seed, run_all,
)

TIER_A = ["SHRED-noise", "SHRED-dropout", "SDN-noise", "SDN-dropout",
          "SenseiverSDN", "RobustSHREDv1", "RobustSHREDv2", "GappyPOD"]
TIER_B = ["SHRED-dropout+mask"]
METHODS = TIER_A + TIER_B
OVERRIDES = ["meeting.methods=[" + ",".join(METHODS) + "]"]
if QUICK:
    OVERRIDES += ["meeting.seeds=[0]", "train.epochs=20", "outputs.root=outputs/meeting_2026-10-14_quick"]
cfg = load_config(OVERRIDES)
ROOT = cfg.outputs.root
if RUN_TRAINING:
    run_all(cfg)
results = load_results(ROOT)
print(f"{len(results)} completed runs in {ROOT}")

{"status": "skipped", "method": "SHRED-noise", "seed": 0}
{"status": "skipped", "method": "SHRED-dropout", "seed": 0}
{"status": "skipped", "method": "SDN-noise", "seed": 0}
{"status": "skipped", "method": "SDN-dropout", "seed": 0}
{"status": "skipped", "method": "SenseiverSDN", "seed": 0}
{"status": "skipped", "method": "RobustSHREDv1", "seed": 0}
{"status": "skipped", "method": "RobustSHREDv2", "seed": 0}
{"status": "skipped", "method": "GappyPOD", "seed": 0}
{"status": "skipped", "method": "SHRED-dropout+mask", "seed": 0}
9 completed runs in outputs/meeting_2026-10-14_quick


## Timing and full-run estimate

In [2]:
rows = ["| Method | wall time (s) | s/epoch | upper bound 1000 epochs (s) | latent stored |", "|---|---:|---:|---:|---|"]
total = 0.0
epochs = int(cfg.train.epochs)
for m in mp.available_methods(results, METHODS):
    r = results[(m, 0)]
    wall = r["wall_time_s"]
    per_epoch = wall / epochs if m != "GappyPOD" else 0.0
    bound = per_epoch * 1000 if m != "GappyPOD" else wall
    total += bound
    has_latent = "p0.3_d_latent" in load_arrays(ROOT, m, 0)
    rows.append(f"| {m} | {wall:.1f} | {per_epoch:.3f} | {bound:.0f} | {has_latent} |")
display(Markdown("\n".join(rows)))
print(f"Upper bound for the full run (3 seeds, all methods hit 1000 epochs): {3 * total / 60:.0f} min")

| Method | wall time (s) | s/epoch | upper bound 1000 epochs (s) | latent stored |
|---|---:|---:|---:|---|
| SHRED-noise | 4.8 | 0.239 | 239 | True |
| SHRED-dropout | 3.4 | 0.172 | 172 | True |
| SDN-noise | 0.7 | 0.033 | 33 | False |
| SDN-dropout | 0.7 | 0.036 | 36 | False |
| SenseiverSDN | 2.6 | 0.132 | 132 | True |
| RobustSHREDv1 | 6.0 | 0.297 | 298 | True |
| RobustSHREDv2 | 4.5 | 0.225 | 225 | True |
| GappyPOD | 0.2 | 0.000 | 0 | False |
| SHRED-dropout+mask | 1.9 | 0.096 | 96 | True |

Upper bound for the full run (3 seeds, all methods hit 1000 epochs): 61 min


## Sanity checks

In [3]:
for m in mp.available_methods(results, METHODS):
    met = results[(m, 0)]["metrics"]
    errs = {c: met[c]["err"] for c in ("clean", "p0", "p0.3")}
    flag = "" if all(np.isfinite(list(errs.values()))) else "  <-- NaN!"
    print(f"{m:20s} " + "  ".join(f"{c}={v:.4f}" for c, v in errs.items()) + flag)

SHRED-noise          clean=0.0886  p0=0.0887  p0.3=0.1695
SHRED-dropout        clean=0.1184  p0=0.1183  p0.3=0.1440
SDN-noise            clean=0.0491  p0=0.0497  p0.3=0.1841
SDN-dropout          clean=0.0753  p0=0.0754  p0.3=0.1201
SenseiverSDN         clean=0.3176  p0=0.3176  p0.3=0.3177
RobustSHREDv1        clean=0.3207  p0=0.3207  p0.3=0.3209
RobustSHREDv2        clean=0.3228  p0=0.3228  p0.3=0.3228
GappyPOD             clean=0.0007  p0=0.0141  p0.3=0.0835
SHRED-dropout+mask   clean=0.1761  p0=0.1758  p0.3=0.1782


**Takeaway:** All 9 methods (Tier A + Tier B) train, evaluate at p = 0 and p = 0.3, and the 6 latent models store latents. The 20-epoch errors are not results (masked models are still at the mean field, ≈ 0.32). The full run (3 seeds, early stopping) took ≈ 16 min on MPS, well below the 61 min upper bound.